In [1]:
"""PET NER dataset download, loading, and pool/test splitting."""

import urllib.request
from pathlib import Path

from config import (
    FEW_SHOT_SPLIT_SEED,
    N_FEW_SHOT_EXAMPLES,
    NER_DATASET_URL,
    NER_TAGS,
    RAW_DATASET_PATH,
    SEED,
    TEST_SIZE,
)
from datasets import ClassLabel, Dataset, Features, Sequence, Value, load_dataset


def download_pet_ner(raw_data_path: Path = RAW_DATASET_PATH, force: bool = False) -> Path:
    """Download the PET entities jsonl into data/raw/."""
    if raw_data_path.exists() and not force:
        return raw_data_path
    raw_data_path.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(NER_DATASET_URL, raw_data_path)
    return raw_data_path


def load_pet_ner(path: Path = RAW_DATASET_PATH) -> Dataset:
    """Load the PET entities dataset (417 sentence-level examples) from data/raw/."""
    if not path.exists():
        raise FileNotFoundError(f"{path} not found — run `uv run uq-pet download-data` first.")
    features = Features(
        {
            "document name": Value("string"),
            "sentence-ID": Value("int8"),
            "tokens": Sequence(Value("string")),
            "ner-tags": Sequence(ClassLabel(names=NER_TAGS)),
        }
    )
    return load_dataset("json", data_files={"full": str(path)}, features=features)["full"]


def split_dataset(
    seed: int = SEED,
    test_size: float = TEST_SIZE,
    n_few_shot: int = N_FEW_SHOT_EXAMPLES,
    few_shot_seed: int = FEW_SHOT_SPLIT_SEED,
) -> tuple[Dataset, Dataset, Dataset]:
    """Split PET into few-shot examples, experiment pool, and held-out test.

    Returns (few_shot, pool, test). The pool excludes the few-shot examples.
    """
    outer = load_pet_ner().train_test_split(test_size=test_size, seed=seed)
    inner = outer["train"].train_test_split(train_size=n_few_shot, shuffle=True, seed=few_shot_seed)
    return inner["train"], inner["test"], outer["test"]


def tag_ids_to_labels(tag_ids: list) -> list[str]:
    return [NER_TAGS[tid] for tid in tag_ids]

In [2]:
print(NER_DATASET_URL, "\n", NER_TAGS, "\n", RAW_DATASET_PATH, "\n", SEED, "\n", TEST_SIZE)

https://raw.githubusercontent.com/patriziobellan86/PETv1.1/master/PETv1.1-entities.jsonl 
 ['O', 'B-Actor', 'I-Actor', 'B-Activity', 'I-Activity', 'B-Activity Data', 'I-Activity Data', 'B-Further Specification', 'I-Further Specification', 'B-XOR Gateway', 'I-XOR Gateway', 'B-Condition Specification', 'I-Condition Specification', 'B-AND Gateway', 'I-AND Gateway'] 
 /Users/mac/Developer/VScode/uq-pet/data/raw/PETv1.1-entities.jsonl 
 3407 
 0.2


In [3]:
download_pet_ner(force=True)

PosixPath('/Users/mac/Developer/VScode/uq-pet/data/raw/PETv1.1-entities.jsonl')

## Prompt creation

In [4]:
from string import Template
from textwrap import dedent

TAG_LEGEND = "\n".join(f"  {i} = {tag}" for i, tag in enumerate(NER_TAGS))

SYSTEM_TEMPLATE = Template(
    dedent("""\
    You are a strict Named Entity Recognition (NER) system for Process Extraction.
    Assign exactly one tag ID to each token in the sentence.

    ENTITY DEFINITIONS:
    - Actor: The person, system, or role performing the action.
    - Activity: The task or action being executed.
    - Activity Data: The object, document, or data manipulated by the activity.
    - Further Specification: Additional context, tools, or locations (e.g., 'via email').
    - XOR Gateway: Words indicating an exclusive branching point (e.g., 'If', 'otherwise').
    - Condition Specification: The condition required to take a branch (e.g., 'the claim is valid').
    - AND Gateway: Words indicating parallel execution (e.g., 'in parallel').
    - O: Tokens outside of any process entity.

    DATASET RULES:
    - Determiners ('The', 'a', 'an') MUST be included in the entity if they precede it.
    - Multi-word entities must start with 'B-' (Beginning) and continue with 'I-' (Inside).
    - Single-word entities get the 'B-' tag.

    TAG IDS:
    $tag_legend

    === EXAMPLES ===
    $examples
    === END OF EXAMPLES ===
    """)
)

USER_TEMPLATE = Template(
    dedent("""\
    Tokens to tag:
    $tokens

    Output MUST be an array of integers with exactly $n_tokens elements, one per
    token, in order. Output nothing except the array.
    """)
)


def build_system_prompt(few_shot: Dataset) -> str:
    """Build the constant prompt prefix — identical for every sentence, so it caches."""
    examples = "\n\n".join(f"{ex['tokens']}\n{ex['ner-tags']}" for ex in few_shot)
    return SYSTEM_TEMPLATE.substitute(tag_legend=TAG_LEGEND, examples=examples)


def build_user_prompt(tokens: list[str]) -> str:
    """Build the user prompt for one sentence of PET tokens."""
    return USER_TEMPLATE.substitute(tokens=tokens, n_tokens=len(tokens))

few_shot, pool, test = split_dataset()

system_prompt = build_system_prompt(few_shot)
pool_messages = [build_user_prompt(ex["tokens"]) for ex in pool]
test_messages = [build_user_prompt(ex["tokens"]) for ex in test]

Generating full split: 0 examples [00:00, ? examples/s]

## UQ using LLMs for NER

In [12]:
print(system_prompt)

You are a strict Named Entity Recognition (NER) system for Process Extraction.
Assign exactly one tag ID to each token in the sentence.

ENTITY DEFINITIONS:
- Actor: The person, system, or role performing the action.
- Activity: The task or action being executed.
- Activity Data: The object, document, or data manipulated by the activity.
- Further Specification: Additional context, tools, or locations (e.g., 'via email').
- XOR Gateway: Words indicating an exclusive branching point (e.g., 'If', 'otherwise').
- Condition Specification: The condition required to take a branch (e.g., 'the claim is valid').
- AND Gateway: Words indicating parallel execution (e.g., 'in parallel').
- O: Tokens outside of any process entity.

DATASET RULES:
- Determiners ('The', 'a', 'an') MUST be included in the entity if they precede it.
- Multi-word entities must start with 'B-' (Beginning) and continue with 'I-' (Inside).
- Single-word entities get the 'B-' tag.

TAG IDS:
  0 = O
  1 = B-Actor
  2 = I-Act

In [5]:
pool_messages[0]

"Tokens to tag:\n['In', 'the', 'meantime', ',', 'the', 'engineering', 'department', 'prepares', 'everything', 'for', 'the', 'assembling', 'of', 'the', 'ordered', 'bicycle', '.']\n\nOutput MUST be a JSON array of integers with exactly 17 elements, one per\ntoken, in order. Output nothing except the array.\n"

In [11]:
pool[0]

{'document name': 'doc-1.1',
 'sentence-ID': 9,
 'tokens': ['In',
  'the',
  'meantime',
  ',',
  'the',
  'engineering',
  'department',
  'prepares',
  'everything',
  'for',
  'the',
  'assembling',
  'of',
  'the',
  'ordered',
  'bicycle',
  '.'],
 'ner-tags': [13, 14, 14, 0, 1, 2, 2, 3, 5, 0, 0, 0, 0, 0, 0, 0, 0]}

### LLM part


In [ ]:
"""Score a split via the NHR@FAU gateway. Threaded, resumable."""
import json
import os
import random
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import openai
from openai import OpenAI
from tqdm.auto import tqdm

MODEL = "RedHatAI/gemma-4-31B-it-FP8-block"
WORKERS = 8

SPLIT = "pool"         # "pool" | "test"
PASS = "dist"          # "dist" | "vote"

messages = {"pool": pool_messages, "test": test_messages}[SPLIT]
dataset = {"pool": pool, "test": test}[SPLIT]

PARAMS = (dict(temperature=1.0, seed=0, n=5, max_tokens=256,
               logprobs=True, top_logprobs=20) if PASS == "dist"
          else dict(temperature=0.8, seed=None, n=10, max_tokens=256))
CACHE = Path(f"data/processed/llm_scores/nhr_gemma_{SPLIT}_{PASS}.jsonl")

client = OpenAI(
    api_key=os.environ["NHR_FAU_API_KEY"],
    base_url="https://hub.nhr.fau.de/api/llmgw/v1",
    timeout=120.0,
    max_retries=0, # do not retry automatically, we handle retries ourselves
)

RETRY = (openai.RateLimitError, 
         openai.APITimeoutError,
         openai.APIConnectionError, 
         openai.InternalServerError)


CACHE.parent.mkdir(parents=True, exist_ok=True)
done = {}
if CACHE.exists():
    with CACHE.open() as f:
        for line in f:
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue                      # truncated tail from a killed run
            done[rec["idx"]] = rec

lock = threading.Lock()
out = CACHE.open("a", buffering=1)


def pack(choice):
    rec = {"text": choice.message.content, "finish_reason": choice.finish_reason}
    content = getattr(choice.logprobs, "content", None) if choice.logprobs else None
    if content:
        rec["logprobs"] = [
            {"token": t.token, "logprob": t.logprob,
             "top": {tp.token: tp.logprob for tp in (t.top_logprobs or [])}}
            for t in content
        ]
    return rec


def score_one(idx):
    for attempt in range(5):
        try:
            r = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "system", "content": system_prompt},
                          {"role": "user", "content": messages[idx]}],
                **PARAMS,
            )
            break
        except RETRY as e:
            if attempt == 4:
                return {"idx": idx, "error": repr(e)}
            time.sleep(min(2 ** attempt, 30) * (0.5 + random.random()))
        except openai.APIStatusError as e:
            return {"idx": idx, "error": repr(e)}        # 400/401/404: fatal

    rec = {"idx": idx, 
           "split": SPLIT,
           "model": r.model, 
           "params": PARAMS,
           "choices": [pack(c) for c in r.choices]}
    with lock:
        out.write(json.dumps(rec) + "\n")
    return rec


todo = [i for i in range(len(messages)) if i not in done]
records = list(done.values())
with ThreadPoolExecutor(max_workers=WORKERS) as ex:
    futures = [ex.submit(score_one, i) for i in todo]
    for fut in tqdm(as_completed(futures), total=len(todo)):
        records.append(fut.result())
out.close()

records.sort(key=lambda r: r["idx"])
failed = [r["idx"] for r in records if "error" in r]
print(f"{SPLIT}/{PASS}: {len(records) - len(failed)} ok, {len(failed)} failed: {failed[:10]}")

In [51]:
for i in range(5):
    print(f"=== {i} ===")
    print(records[4]["choices"][i]["text"])

=== 0 ===
[5, 6, 0, 0, 3, 7, 8, 8, 0]
=== 1 ===
[5, 6, 0, 0, 3, 7, 8, 8, 0]
=== 2 ===
[5, 6, 0, 0, 3, 7, 8, 8, 0]
=== 3 ===
[5, 6, 0, 0, 3, 7, 8, 8, 0]
=== 4 ===
[5, 6, 0, 0, 3, 7, 8, 8, 0]


### UQ part
